# Day 72: Word Embeddings

## The Problem with Raw Text
Neural networks only understand numbers, so raw text has to be converted into a numeric form before it can be fed into any model. A naive approach - like one-hot encoding every word - creates huge, sparse vectors that treat every word as completely unrelated to every other word, which wastes memory and throws away useful information (e.g. that "happy" and "joyful" are related). Word embeddings solve this by representing each word as a dense, relatively low-dimensional vector of real numbers (e.g. 8 or 100 dimensions instead of "vocabulary size" dimensions), where words with similar meaning or usage end up with similar vectors. The code below is the first step of that pipeline: `Tokenizer` scans a small set of example sentences, assigns each unique word an integer ID, and `texts_to_sequences` converts each sentence into a list of those integer IDs in order - this integer sequence is what later gets mapped to embedding vectors.</cell id="cell-1">


In [ ]:
# pip install tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

texts = ['I love data science', 'Data science is fun', 'I love learning AI', 'AI is the future']
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)  # builds the word -> integer index vocabulary from these sentences
sequences = tokenizer.texts_to_sequences(texts)  # converts each sentence into a list of word-index integers
print(sequences)

## Padding Sequences
Sentences are naturally different lengths, but layers like Embedding and Dense expect every input in a batch to have the same fixed shape. `pad_sequences` handles this by adding zeros (a reserved "no word" index) to the front of shorter sequences until they all reach the same length - `maxlen=6` here - and truncating anything longer than that. Padding at the front (the default) rather than the end is a common convention for RNN-style models, since it keeps the actual content aligned toward the end of the sequence. This padded, fixed-length integer array is what actually gets fed into the model as input.</cell id="cell-3">


In [ ]:
padded = pad_sequences(sequences, maxlen=6)  # pad/truncate every sequence to exactly 6 tokens (zeros added at the front by default)
print(padded)

## The Embedding Layer
The `Embedding` layer is essentially a lookup table with `vocab_size` rows (one per word in the vocabulary) and `output_dim` columns (the size of each word's vector) - given a word's integer index, it returns that row as the word's dense vector representation. Unlike a fixed lookup table, these vectors start out random and are learned during training just like any other weights, so the model adjusts them to place words that behave similarly (e.g. appear in similar contexts) closer together in the vector space. `input_length=6` matches the padded sequence length from the previous step, and `Flatten` turns the resulting 2D (timesteps x embedding_dim) block into a 1D vector so it can feed into the Dense output layer - a simpler alternative to feeding it through an RNN/LSTM. This model is just for illustrating the embedding layer's shape; it isn't trained here (no compile/fit), so `model.summary()` is only showing the architecture and parameter counts.</cell id="cell-5">


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

vocab_size = len(tokenizer.word_index) + 1  # +1 because index 0 is reserved for padding, not an actual word
model = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=8, input_length=6),  # learns an 8-dim vector for each word in the vocabulary
    layers.Flatten(),  # collapse (timesteps, embedding_dim) into a single vector for the Dense layer
    layers.Dense(1, activation='sigmoid')
])
model.summary()

## Pretrained Embeddings
Learning embeddings from scratch (as in the cell above) requires enough training data for the model to figure out good word representations on its own, which is often unrealistic for small datasets. Pretrained embeddings solve this by reusing word vectors that were already learned from massive text corpora (billions of words) - Word2Vec and GloVe learn embeddings by looking at which words tend to appear near each other, while FastText extends this to also use sub-word information (useful for rare words or typos). Plugging these pretrained vectors into the Embedding layer (either frozen or fine-tuned) gives a small project a huge head start, since the model starts with vectors that already encode real semantic relationships rather than random noise. More recently, transformer-based models (like BERT) go a step further by producing embeddings for entire sentences that account for context - the same word can get a different vector depending on the sentence it's in - which is why sentence embeddings from transformers are now the more common choice for state-of-the-art NLP work, though word embeddings remain useful, faster, and simpler for many smaller tasks.</cell id="cell-7">


In [ ]:
print('Pretrained options: Word2Vec, GloVe, FastText')
print('Modern alternative: sentence embeddings from transformer models')